In [ ]:

import pandas as pd
import random
import numpy as np
from tqdm import tqdm
import torch
import reverse_geocoder as rg
from scipy.sparse import csr_matrix
import scipy.sparse as sp
from pathlib import Path
import sys
import os
parent_dir = os.path.join(os.getcwd(), '..')
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
# Assuming these imports are available in your environment
from model.pred import (
    LightGCNModel, MFCF,
    CFItemPrecomputed, 
    CFUserPrecomputed,
    GPT2Recommender,
)
from model.proposed_model import MaskedPOIGPTRecommender
from data_utils.utils import per_user_last_split, dict_to_lists
from data_utils.sampling import preprocess_seq
from eval.losses import ndcg_at_k
# from model.proposed_model import x

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", DEVICE)

# %%
# ===== Cell 2 — Configuration & constants =====
# Dataset configurations
DATASETS = [
    {
        'name': 'NYC',
        'path': '../data/TSMC2014_nyc.parquet',
        'min_points': 50,
        'test_frac': 0.2
    },
    {
        'name': 'TKY',
        'path': '../data/TSMC2014_tky.parquet',
        'min_points': 50,
        'test_frac': 0.2
    }
]

# Model hyperparameters
MF_EMB_DIM = 32
MF_EPOCHS = 10
MF_LR = 3e-4

GCN_LAYERS = 2
GCN_EMB_DIM = 64
GCN_EPOCHS = 10
GCN_LR = 1e-2

GPT_EPOCHS = 100
GPT_BATCH_SIZE = 16

# Evaluation settings
K_VALUES = [5, 10]

# General settings
RESULTS_DIR = "results"

# Dataset columns (for TSMC files)
COLS = [
    "user_id", "venue_id", "category_id", "category",
    "lat", "lon", "tz_offset_min", "utc_time"
]

# %%
# ===== Cell 3 — Results directory & containers =====
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
all_results = []
all_summaries = []

print("Starting recommendation system experiment...")


In [ ]:
final_result = {}
for dataset in DATASETS:
    
    dataset_name = dataset['name']
    dataset_path = dataset['path']
    min_points = dataset['min_points']
    test_frac = dataset['test_frac']

    print(f"\n{'='*60}")
    print(f"Processing dataset: {dataset_name}")
    print(f"{'='*60}")

    # --- Load dataset (inline; no helper func) ---
    print(f"Loading dataset from {dataset_path}")
    df = pd.read_parquet(
        dataset_path,
    )

    print(f"Loaded {len(df)} records")

    # --- Filter sparse users (inline) ---
    print(f"Filtering users with < {min_points} points...")
    counts = df["user_id"].value_counts()
    df_filtered = df[df["user_id"].map(counts) >= min_points].copy()

    # Cap to first min_points per user (by time ascending)
    df_filtered = df_filtered.sort_values(["user_id", "utc_time"], ascending=[True, True])
    df_filtered = df_filtered.groupby("user_id", group_keys=False).head(min_points)

    print(f"Kept {len(df_filtered)} records from {df_filtered['user_id'].nunique()} users")

    # Keep needed columns
    df_filtered = df_filtered[['user_id', 'category', 'lat', 'lon', 'utc_time']].reset_index(drop=True)

    # --- Per-user temporal split ---
    train_df, test_df = per_user_last_split(
        df_filtered, user_col='user_id', time_col='utc_time', test_frac=test_frac
    )

    # --- Build mappings & interaction matrices (inline) ---
    users = df_filtered['user_id'].unique()
    items = df_filtered['category'].unique()
    user2idx = {u: i for i, u in enumerate(users)}
    item2idx = {i: j for j, i in enumerate(items)}
    idx2item = {j: i for i, j in item2idx.items()}
    n_users, n_items = len(users) + 1, len(items) + 1  # +1 for safety

    def transform(data):
        new_data = [item2idx[ele] for ele in data]
        return new_data
    # Helper-free split builder for train
    grp_train = train_df.groupby('user_id')["category"].agg(list).reset_index()
    R_train = sp.lil_matrix((n_users, n_items), dtype=int)
    train_list = []
    for _, row in grp_train.iterrows():
        u = row['user_id']
        seq_idx = [item2idx[i] for i in row['category'] if i in item2idx]
        train_list.append(seq_idx)
        if u in user2idx and len(seq_idx) > 0:
            R_train[user2idx[u], seq_idx] = 1

    # Same for test
    grp_test = test_df.groupby('user_id')["category"].agg(list).reset_index()
    R_test = sp.lil_matrix((n_users, n_items), dtype=int)
    test_list = []
    for _, row in grp_test.iterrows():
        u = row['user_id']
        seq_idx = [item2idx[i] for i in row['category'] if i in item2idx]
        test_list.append(seq_idx)
        if u in user2idx and len(seq_idx) > 0:
            R_test[user2idx[u], seq_idx] = 1

    # Sequence dicts (raw) → preprocessed (index-based) via your utility
    train_seqs = train_df.groupby('user_id')['category'].agg(list).to_dict()
    test_seqs = test_df.groupby('user_id')["category"].agg(list).to_dict()
    train_seqs = preprocess_seq(train_seqs, user2idx, item2idx)
    test_seqs = preprocess_seq(test_seqs, user2idx, item2idx)
    all_items_for_predict = list(item2idx.keys())

    # --- Train models (inline) ---
    print("Training recommendation models...")
    models = {}
    vocab_size = list(item2idx.values())[-1] + 1
    train_vals = list(train_seqs.values())
    train_keys = list(train_seqs.keys())
    print("- Training Matrix Factorization...")
    mf = MFCF(n_users, n_items, emb_dim=MF_EMB_DIM, device=DEVICE)
    mf.fit(
        train_df, user2idx, item2idx, R_train,
        epochs=MF_EPOCHS, lr=MF_LR, loc_col='category', user_col='user_id'
    )
    models['mf'] = mf

    # LightGCN
    print("- Training LightGCN...")
    pos_u = torch.tensor([user2idx[u] for u in train_df['user_id']], dtype=torch.long)
    pos_i = torch.tensor([item2idx[i] for i in train_df['category']], dtype=torch.long)
    edge_index = torch.stack([
        torch.cat([pos_u, pos_i + n_users]),
        torch.cat([pos_i + n_users, pos_u])
    ], dim=0)
    
    gcn = LightGCNModel(n_users, n_items, layers=GCN_LAYERS, emb_dim=GCN_EMB_DIM, device=DEVICE)
    gcn.fit(pos_u, pos_i, edge_index, epochs=GCN_EPOCHS, lr=GCN_LR)
    models['gcn'] = gcn

    # CF (Item-based)
    print("- Training Item-based CF...")
    cf_item = CFItemPrecomputed(n_users=n_users, n_items=n_items)
    cf_item.fit(csr_matrix(R_train), n_jobs=-1)
    models['cf_item'] = cf_item

    # CF (User-based)
    print("- Training User-based CF...")
    cf_user = CFUserPrecomputed(n_users=n_users, n_items=n_items)
    cf_user.fit(csr_matrix(R_train), n_jobs=-1)
    models['cf_user'] = cf_user

    # GPT2
    print("- Training GPT2...")
    gpt = GPT2Recommender(device=DEVICE)
    train_keys, train_vals = dict_to_lists(train_seqs)

    gpt.fit(train_vals, epochs=GPT_EPOCHS, batch_size=GPT_BATCH_SIZE)
    models['gpt'] = gpt

    recommender = MaskedPOIGPTRecommender(
        hidden_dim=384,
        num_layers=6, 
        num_heads=6,
        max_length=128
    )
    recommender.fit(train_df, epochs=100, batch_size=32, lr=5e-5)
    prediction = recommender.predict(n_tokens=20)
    prediction_transformed = {
        key: [item2idx[ele] for ele in val] 
        for key, val in prediction.items()
    }
    prediction = recommender.predict(n_tokens=1)
    eval_rows = []
    with torch.inference_mode():
        for K in K_VALUES:
            print(f"Evaluating at K={K}")
            for uid, ori_true_seq in tqdm(test_seqs.items(), desc=f"nDCG@{K}"):
                if uid not in user2idx or not ori_true_seq:
                    continue
                true_seq = ori_true_seq[:K]
                mf_pred = transform(models['mf'].predict(uid, all_items_for_predict, user2idx, item2idx, reorder_model=None)[:K])
            
                # GCN
                gcn_pred = transform(models['gcn'].predict(uid, all_items_for_predict, user2idx, item2idx, reorder_model=None)[:K])
        
                # CF (User)
                ucf_pred = transform(models['cf_user'].predict(
                    u_id=uid, i_ids=all_items_for_predict, user2idx=user2idx,
                    item2idx=item2idx, idx2item=idx2item, reorder_model=None
                )[:K])
        
                # CF (Item)
                icf_pred = transform(models['cf_item'].predict(
                    u_id=uid, i_ids=all_items_for_predict, user2idx=user2idx,
                    item2idx=item2idx, idx2item=idx2item, reorder_model=None
                )[:K])
            
                # GPT2
                try:
                    input_seq = torch.tensor([[ele for ele in train_seqs[uid]]],
                                                dtype=torch.long, device=DEVICE)
                    gpt_out = models['gpt'].predict(input_seq, num_new_tokens=K, device=DEVICE)[0][len(train_seqs[uid]):]
                    gpt_ndcg = ndcg_at_k(gpt_out, true_seq, k=K)
                except Exception as e:
                     print(f"GPT2 prediction failed for user {uid}: {e}")
                    gpt_ndcg = 0.0
                eval_rows.append({
                    'uid': uid,
                    'K': K,
                    'true_seq': true_seq,
                    
                    'MF_pred': mf_pred,
                     'GCN_pred': gcn_pred,
                    'UserCF_pred': ucf_pred,
                    'ItemCF_pred': icf_pred,
                    'GPT2_pred': gpt_out,
                    "proposed_model_pred": prediction_transformed[uid][:K],

                    'MF_nDCG': ndcg_at_k(mf_pred, true_seq, k=K),
                    'GCN_nDCG': ndcg_at_k(gcn_pred, true_seq, k=K),
                    'UserCF_nDCG': ndcg_at_k(ucf_pred, true_seq, k=K),
                    'ItemCF_nDCG': ndcg_at_k(icf_pred, true_seq, k=K),
                    'GPT2_nDCG': gpt_ndcg,
                    'proposed_model_nDCG': ndcg_at_k(prediction_transformed[uid], true_seq , k=K)
                })
    final_result[dataset_name] = eval_rows